# Message Protocol
# 0. 介绍
**研究背景**：Agent 会把大模型与用户、工具之间的每次交流保存成消息历史。为了判断两份消息历史是否相同，程序通常会先把消息转成 JSON 文本，再计算一个像“数字身份证”一样的指纹。

**现存问题**：同一份消息中的`字段`可能出现`顺序不同`，`含义完全相同`，但普通 JSON 会生成不同的文本和指纹。这样，同一次对话可能被误认为两次不同的请求，造成缓存失效、重复执行和重复计费。

**解决方案**：本 Notebook 将实现一个极简的 Message Protocol，用 **Canonical JSON (规范 JSON)** 统一字段顺序和多余空格。然后用同一份真实 API 消息历史进行对比：基线版本产生不同指纹，改进版本产生相同指纹，从而直观看到稳定的消息协议如何避免重复执行。
## 目录
0. 介绍
1. 初始化真实 API
2. 前置准备
3. 获取并验证 API 响应
4. 定义基线组件 *
5. 展示基线故障 *
6. 定义改进组件 *
7. 展示修复结果 *
8. 汇总消融对照

# 1. 初始化真实 API
## 连接大模型
程序需要先读取项目 `.env` 文件中已经准备好的连接信息，才能使用真实的大模型。本节直接读取这些信息并建立连接，同时保存后面要使用的模型名称。

In [1]:
from dotenv import dotenv_values, find_dotenv
from openai import OpenAI

config = dotenv_values(find_dotenv())  # 自动找到并读取项目的 .env
client = OpenAI(
    api_key=config["OPENAI_API_KEY"],
    base_url=config["OPENAI_BASE_URL"],
)
model_name = config["OPENAI_MODEL"]
print(f"真实 API 已就绪：{model_name}")

真实 API 已就绪：LongCat-2.0


输出显示了模型名称，说明真实 API 已经准备好，但此时还没有向大模型发送请求。下一章将定义大模型要完成的任务，以及程序接收和处理结果的方式。

# 2. 前置准备
## 2.1 说明可用工具
消息历史需要包含大模型返回的操作，才能测试真实的消息协议。本节用一份简单的说明告诉大模型，它可以使用 `get_weather` 查询城市天气。

In [2]:
tools = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "查询城市天气",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string"}},
            "required": ["city"],
        }}}]
print(f"可用工具：{tools[0]['function']['name']}")

可用工具：get_weather


输出显示 `get_weather` 已经准备好，说明大模型知道了工具名称和需要填写的城市。下一步会给大模型一个具体问题，让它返回一条真实的工具调用消息。

## 2.2 写出具体任务
有了工具说明，还需要告诉大模型这次要做什么。本节询问北京天气，并明确要求大模型使用刚才的工具回答。

In [3]:
messages = [
    {"role": "system", "content": "你是天气助手，只能调用 get_weather。"},
    {"role": "user", "content": "请查询北京今天的天气。"},
]
print(f"任务：{messages[-1]['content']}")

任务：请查询北京今天的天气。


输出显示了大模型将要处理的问题。下一步会定义统一的指纹计算方式，把任意一段 JSON 文本变成便于比较的短标识。

## 2.3 定义指纹计算方式
直接比较很长的 JSON 文本不够直观。本节使用 SHA-256 把文本变成固定长度的指纹；文本只要有一个字符不同，指纹通常就会不同。

In [4]:
import hashlib

def fingerprint(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

print("指纹计算方式：SHA-256")

指纹计算方式：SHA-256


输出说明指纹函数已经准备好，但目前还没有文本流入。下一步会定义一种只改变字段顺序、不改变消息内容的方式，用来制造对照数据。

## 2.4 定义字段换序方式
为了测试消息协议是否稳定，需要准备一份含义相同但字段顺序不同的消息历史。本节反转每条消息的字段顺序，消息中的内容保持不变。

In [5]:
# def reverse_fields(messages):
#     return [{key: message[key] for key in reversed(message)} for message in messages]
def reverse_fields(messages):
    reversed_messages = []
    for message in messages:
        reversed_msg = {}
        for key in reversed(message):
            reversed_msg[key] = message[key]
        reversed_messages.append(reversed_msg)
    return reversed_messages
    
print("换序方式：反转每条消息的字段顺序")

换序方式：反转每条消息的字段顺序


输出说明字段换序函数已经准备好。它只改变字段出现的先后，不增删任何数据；下一步会定义判断消息协议是否成功的统一标准。

## 2.5 定义成功标准
两份消息历史的内容相同，只是字段顺序不同，因此稳定的消息协议应该为它们生成相同指纹。本节把这个要求写成一个简单判断，后面两种做法都会使用同一标准。

In [6]:
def grade(first_fingerprint, second_fingerprint):
    return first_fingerprint == second_fingerprint

print("成功标准：字段换序前后的指纹相同")

成功标准：字段换序前后的指纹相同


输出显示了唯一的成功标准，说明后面的结果可以用同一把尺子判断。至此，工具、任务、指纹、字段换序方式和成功标准都已准备完成，下一章将调用真实大模型并查看它返回的原始消息。

# 3. 获取并验证 API 响应
## 3.1 获取真实响应
程序已经准备好模型、工具和任务，现在可以把它们一起发给大模型。本节要求大模型必须选择工具，并记录从发出请求到收到回复所用的时间。

In [7]:
import json
from time import perf_counter

start = perf_counter()
response = client.chat.completions.create(
    model=model_name,
    messages=messages,
    tools=tools,
    tool_choice="required",  # 必须选择一个工具
    temperature=0,
)
latency_ms = round((perf_counter() - start) * 1000)
raw_real = response.model_dump()
print(f"回复已收到：provider={config['NANO_BACKEND']}, model={model_name}, latency={latency_ms} ms")
print("具体内容如下：")
print(json.dumps(raw_real, indent=4, ensure_ascii=False))

回复已收到：provider=openai, model=LongCat-2.0, latency=4009 ms
具体内容如下：
{
    "id": "caada8e3568b41bca2d3f09d3a95e9c3",
    "choices": [
        {
            "finish_reason": "tool_calls",
            "index": 0,
            "logprobs": null,
            "message": {
                "content": "我来帮您查询北京今天的天气情况。",
                "refusal": null,
                "role": "assistant",
                "annotations": null,
                "audio": null,
                "function_call": null,
                "tool_calls": [
                    {
                        "id": "call_2dfc1955a8654c0090656806",
                        "function": {
                            "arguments": "{\"city\": \"北京\"}",
                            "name": "get_weather"
                        },
                        "type": "function",
                        "index": null
                    }
                ],
                "reasoning_content": "\n用户要求查询北京今天的天气。我需要使用 get_weather 函数来查询北京的天气信息。根据函数定义，我需要

输出显示了模型来源、模型名称和等待时间，说明真实大模型已经返回结果，完整内容保存在 `raw_real` 中。下一步会打开这份原始结果，确认大模型选择了哪个工具并填写了什么内容。

## 3.2 查看并保存响应
大模型已经返回结果，但我们还不知道它是否正确调用了工具。本节读取工具名称、参数、停止原因和 Token 用量，并把真实回复接到已有消息后，形成后续实验使用的完整消息历史。

In [8]:
choice = raw_real["choices"][0]
assistant_message = choice["message"]
tool_call = assistant_message["tool_calls"][0]
arguments = json.loads(tool_call["function"]["arguments"])
real_messages = messages + [assistant_message]
print(f"工具：{tool_call['function']['name']}")
print(f"参数：{arguments}")
print(f"停止原因：{choice['finish_reason']}")
print(f"Token 用量：{raw_real['usage']}")
print(f"消息数量：{len(real_messages)}")

工具：get_weather
参数：{'city': '北京'}
停止原因：tool_calls
Token 用量：{'completion_tokens': 59, 'prompt_tokens': 152, 'total_tokens': 211, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 33, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 128, 'image_tokens': 0, 'video_tokens': 0, 'text_tokens': 0}, 'effectiveCachedTokens': 128, 'cache_write_tokens': 0, 'cache_read_tokens': 0, 'input_tokens': 0, 'output_tokens': 0, 'output_tokens_details': None, 'cached_tokens': 0}
消息数量：3


输出显示大模型调用了 `get_weather` 并填写了城市，说明真实回复可以用于实验。`real_messages` 依次保存 system、user 和 assistant 消息；停止原因、Token 用量与延迟共同记录了本次数据的来源和成本。下一章将定义普通 JSON 基线组件。

# 4. 定义基线组件
## 4.1 定义普通 JSON 序列化器
消息历史必须先变成 JSON 文本，才能计算指纹。最简单的做法是直接使用普通 JSON，它会保留字段原来的顺序；本节先把这种不稳定的做法保留为基线。

In [9]:
class PlainJson:
    def dumps(self, messages):
        return json.dumps(messages, ensure_ascii=False)  # 保留字段原顺序

baseline = PlainJson()
print("基线组件：普通 JSON，不统一字段顺序")

基线组件：普通 JSON，不统一字段顺序


输出说明基线组件已经定义完成，它会把消息历史转成 JSON 文本，但不会统一字段顺序。下一章将让它处理第 3 章保存的真实消息，查看字段换序前后的指纹是否一致。

# 5. 展示基线故障
## 5.1 序列化真实消息
第 3 章已经保存了真实消息，现在先反转每条消息的字段顺序，再让普通 JSON 分别处理换序前后的数据。本节同时比较 Python 数据和 JSON 文本，确认内容是否真的发生变化。

In [10]:
reordered_messages = reverse_fields(real_messages)
baseline_original = baseline.dumps(real_messages)
baseline_reordered = baseline.dumps(reordered_messages)
print(f"消息内容相同：{real_messages == reordered_messages}")
print(f"原始文本：{baseline_original[:120]}")
print(f"换序文本：{baseline_reordered[:120]}")

消息内容相同：True
原始文本：[{"role": "system", "content": "你是天气助手，只能调用 get_weather。"}, {"role": "user", "content": "请查询北京今天的天气。"}, {"content": "我来帮
换序文本：[{"content": "你是天气助手，只能调用 get_weather。", "role": "system"}, {"content": "请查询北京今天的天气。", "role": "user"}, {"reasoning_cont


输出中的 `消息内容相同：True` 说明两份数据表达同一段对话，但两段 JSON 文本的字段顺序不同。下一步会分别计算它们的指纹，查看这点文本差异会造成什么结果。

## 5.2 计算基线指纹
JSON 文本不同是否重要，要看它们生成的指纹。本节使用第 2 章定义的同一个 SHA-256 函数，分别计算原始文本和换序文本的指纹。

In [11]:
baseline_fp_original = fingerprint(baseline_original)
baseline_fp_reordered = fingerprint(baseline_reordered)
print(f"原始指纹：{baseline_fp_original}")
print(f"换序指纹：{baseline_fp_reordered}")

原始指纹：be60644a2a76718455c1ccf7097c08ce1f77c437288a4e72a41720d7a9219d21
换序指纹：fba1631a1138cc9afeef70f545d99a95654ea2ea95536a3143ff0c44b4082a12


输出显示两串指纹完全不同。普通 JSON 把字段顺序写进了文本，因此只改变字段出现的先后，就会让缓存或审计系统误以为遇到了另一段消息；下一步将用统一标准给出最终结论。

## 5.3 判断基线结果
两份消息内容相同，却得到了不同指纹，还需要用第 2 章的成功标准明确判断。本节比较两个指纹，并保存基线做法的最终结果。

In [12]:
baseline_passed = grade(baseline_fp_original, baseline_fp_reordered)
print(f"基线任务通过：{baseline_passed}")

基线任务通过：False


输出为 `False`，说明基线做法没有通过。真实消息的内容没有改变，但普通 JSON 生成了不同指纹；下一章将定义一个能够统一字段顺序的改进组件。

# 6. 定义改进组件
## 6.1 定义 Canonical JSON 序列化器
基线失败是因为普通 JSON 保留了字段原来的顺序。改进方法是先递归排序所有字段，再移除多余空格，使相同数据得到相同文本；这是 Canonical JSON 的最小核心，不包含 RFC 8785 的全部边界规则。

In [13]:
class CanonicalJson:
    def dumps(self, messages):
        return json.dumps(messages, ensure_ascii=False, sort_keys=True, separators=(",", ":"))

fixed = CanonicalJson()
print("改进组件：Canonical JSON，统一字段顺序和空格")

改进组件：Canonical JSON，统一字段顺序和空格


输出说明改进组件已经定义完成。`sort_keys=True` 会统一所有层级的字段顺序，`separators` 会移除多余空格；下一章将让它处理与基线完全相同的两份真实消息。

# 7. 展示修复结果
## 7.1 序列化真实消息
为了只比较序列化方式，本节继续使用第 5 章的原始消息和字段换序消息。唯一变化是把普通 JSON 换成 Canonical JSON，然后观察两段文本是否仍有差异。

In [14]:
fixed_original = fixed.dumps(real_messages)
fixed_reordered = fixed.dumps(reordered_messages)
print(f"规范文本相同：{fixed_original == fixed_reordered}")
print(f"原始文本：{fixed_original[:120]}")
print(f"换序文本：{fixed_reordered[:120]}")

规范文本相同：True
原始文本：[{"content":"你是天气助手，只能调用 get_weather。","role":"system"},{"content":"请查询北京今天的天气。","role":"user"},{"annotations":null,"aud
换序文本：[{"content":"你是天气助手，只能调用 get_weather。","role":"system"},{"content":"请查询北京今天的天气。","role":"user"},{"annotations":null,"aud


输出中的 `规范文本相同：True` 说明 Canonical JSON 消除了字段顺序差异，两份相同内容现在得到完全相同的文本。下一步会计算指纹，确认文本修复是否传递到了指纹层。

## 7.2 计算改进指纹
文本已经相同，还需要使用与基线相同的 SHA-256 函数计算指纹。本节分别计算两段规范文本的指纹，查看字段换序是否还能造成漂移。

In [15]:
fixed_fp_original = fingerprint(fixed_original)
fixed_fp_reordered = fingerprint(fixed_reordered)
print(f"原始指纹：{fixed_fp_original}")
print(f"换序指纹：{fixed_fp_reordered}")

原始指纹：5e7f2aba848669dc2361b265454a45800dea868658a0e8083572e54a96d23e71
换序指纹：5e7f2aba848669dc2361b265454a45800dea868658a0e8083572e54a96d23e71


输出显示两串指纹完全相同，说明字段顺序不再影响缓存键或审计指纹。下一步将使用与基线相同的成功标准判断改进任务是否通过。

## 7.3 判断改进结果
指纹已经相同，还需要用统一标准给出明确结论。本节仍然比较两个指纹，并保存改进做法的最终结果。

In [16]:
fixed_passed = grade(fixed_fp_original, fixed_fp_reordered)
print(f"改进任务通过：{fixed_passed}")

改进任务通过：True


输出为 `True`，说明改进组件成功解决了指纹漂移。两种做法使用同一个模型响应、字段换序方式、指纹函数和成功标准，结果差异只来自 JSON 序列化方式；下一章将汇总完整对照。

# 8. 汇总消融对照
## 8.1 对比两种做法
只改变序列化组件并比较前后结果，就能看出 Message Protocol 是否重要。本节先汇总共同使用的真实 API 响应，再并排记录两种做法的字节数、指纹稳定性和字段顺序敏感性。

In [17]:
baseline_bytes = len(baseline_original.encode("utf-8"))
fixed_bytes = len(fixed_original.encode("utf-8"))
shared_info = {"模型来源": config["NANO_BACKEND"], "模型": model_name, "等待时间（毫秒）": latency_ms, "Token 总量": raw_real["usage"]["total_tokens"], "停止原因": choice["finish_reason"], "消息数量": len(real_messages)}
comparison = [
    {"做法": "普通 JSON", "字节数": baseline_bytes, "指纹稳定": baseline_passed, "字段顺序敏感": not baseline_passed},
    {"做法": "Canonical JSON", "字节数": fixed_bytes, "指纹稳定": fixed_passed, "字段顺序敏感": not fixed_passed},
]
print("共同信息：")
print(json.dumps(shared_info, ensure_ascii=False, indent=2))
print("消融对照：")
print(json.dumps(comparison, ensure_ascii=False, indent=2))

共同信息：
{
  "模型来源": "openai",
  "模型": "LongCat-2.0",
  "等待时间（毫秒）": 4009,
  "Token 总量": 211,
  "停止原因": "tool_calls",
  "消息数量": 3
}
消融对照：
[
  {
    "做法": "普通 JSON",
    "字节数": 678,
    "指纹稳定": false,
    "字段顺序敏感": true
  },
  {
    "做法": "Canonical JSON",
    "字节数": 645,
    "指纹稳定": true,
    "字段顺序敏感": false
  }
]


输出显示两种做法使用同一份真实 API 消息：普通 JSON 对字段顺序敏感，指纹不稳定，任务失败；Canonical JSON 对字段顺序不敏感，指纹稳定，任务通过。模型没有改变，决定结果的是模型外层的消息协议，至此本 Notebook 的对照实验结束。

## 8.2 拓展

### nano 版省略了什么

nano 版只演示少量消息字段的确定性 JSON，没有覆盖协议版本协商、二进制附件、流式增量、跨进程签名、Schema 演进、向后兼容、幂等请求 ID 和敏感字段脱敏。生产协议还需要明确错误模型与能力协商；这些能力不会改变本例的不变量：同一语义消息必须得到稳定、可验证的线表示。

### 延伸阅读


1. 2024, [Anthropic, Introducing the Model Context Protocol](https://www.anthropic.com/news/model-context-protocol)：统一消息与能力边界如何替代一次性私有连接。
2. 2025, [Model Context Protocol Specification 2025-06-18](https://modelcontextprotocol.io/specification/2025-06-18)：JSON-RPC 消息、生命周期、能力协商和错误语义。
3. 2025, [Google, Announcing the Agent2Agent protocol](https://developers.googleblog.com/en/a2a-a-new-era-of-agent-interoperability/)：跨 Agent 任务、消息与产物互操作协议。